This notebook is used to create zip files that are used for archiving and uploading data to the S3 bucket.

In [ ]:
%reload_ext autoreload
%autoreload 2

# # Tell python where to look for modules.
import sys
import os

sys.path.append("../../src")

import oge.output_data as output_data
import oge.filepaths as filepaths

In [ ]:
# build the data manifest
output_data.build_data_download_manifest(skip_outputs=False)

In [ ]:
# zip all data for s3 and zenodo
earliest_year = 2005
latest_year = 2025
years = list(range(earliest_year, latest_year+1))

for year in years:
    output_data.zip_data_for_zenodo(year)

# Delete outputs/results that we don't need locally
Saves space on computer

In [ ]:
# save local disk space by removing output files that are not generally used except to upload
# results: remove all metric unit files
# outputs: remove shaped_eia data, hourly profiles, cems_cleaned
import glob, os, shutil

glob.glob(filepaths.results_folder("*/metric_units"))

In [ ]:
# remove metric unit folders
def fast_scandir(dirname):
    subfolders = [f.path for f in os.scandir(dirname) if f.is_dir()]
    for dirname in list(subfolders):
        subfolders.extend(fast_scandir(dirname))
    return subfolders


# find the folders to remove
metric_unit_folders = fast_scandir(filepaths.results_folder())
metric_unit_folders = [dir for dir in metric_unit_folders if "metric_units" in dir]
# remove the folder
for folder in metric_unit_folders:
    shutil.rmtree(folder)

In [ ]:
# remove specified output files
outputs_to_remove = ["cems_cleaned", "hourly_profiles", "shaped_eia923_data"]
output_years = [f.path for f in os.scandir(filepaths.outputs_folder()) if f.is_dir()]
# create a list of all of the files to remove
files_to_remove = []
for dir in output_years:
    output_files = [f.path for f in os.scandir(dir)]
    for output in outputs_to_remove:
        files_to_remove += [f for f in output_files if output in f]
# remove files
for file in files_to_remove:
    os.remove(file)